# Chapter 3 Source Code Notebook

This notebook builds the source code for Chapter 3, **Memory, World Models, and Learning**.

The code builds structured reasoning with state machines, memory management, time budgets, logging, and traceable execution for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: The State Machine Controller

### `reasoning/state_machine.py`

The chapter begins by turning workflow progress into explicit states and transitions. This gives the reasoning process a controlled path instead of leaving progress to the model alone.

In [ ]:
# reasoning/state_machine.py
from dataclasses import dataclass
from typing import Callable, Optional
from ..session import Session, SessionStatus

GuardFn  = Callable[[Session], tuple[bool, str]]
ActionFn = Callable[[Session], None]

@dataclass
class Transition:
    from_state: SessionStatus
    to_state:   SessionStatus
    guard:      Optional[GuardFn]  = None   # must return (True, '') to proceed
    on_enter:   Optional[ActionFn] = None   # runs after transition

class StateMachine:
    def __init__(self, initial: SessionStatus):
        self.transitions: list[Transition] = []
        self.initial = initial

    def add_transition(self, t: Transition) -> 'StateMachine':
        self.transitions.append(t)
        return self

    def transition(self, session: Session,
                   target: SessionStatus) -> tuple[bool, str]:
        """Attempt a transition. Returns (success, reason)."""
        valid = [
            t for t in self.transitions
            if t.from_state == session.status and t.to_state == target
        ]
        if not valid:
            return False, f"No transition: {session.status} -> {target}"

        t = valid[0]
        if t.guard:
            passed, reason = t.guard(session)
            if not passed:
                return False, f"Guard failed: {reason}"

        session.status = target
        if t.on_enter:
            t.on_enter(session)
        return True, "ok"

# ── Build the SupportOps state machine ────────────────────────────────
def guard_routing_complete(session: Session) -> tuple[bool, str]:
    if not session.routing_category:
        return False, "routing_category not set"
    return True, ""

def guard_classification_complete(session: Session) -> tuple[bool, str]:
    if session.classification is None:
        return False, "classification not set"
    return True, ""

def guard_draft_complete(session: Session) -> tuple[bool, str]:
    if not session.draft_response:
        return False, "draft_response is empty"
    if len(session.draft_response) < 20:
        return False, "draft_response too short to be valid"
    return True, ""

def build_support_state_machine() -> StateMachine:
    sm = StateMachine(initial=SessionStatus.OPEN)
    sm.add_transition(Transition(
        SessionStatus.OPEN, SessionStatus.ROUTING))
    sm.add_transition(Transition(
        SessionStatus.ROUTING, SessionStatus.PROCESSING,
        guard=guard_routing_complete))
    sm.add_transition(Transition(
        SessionStatus.ROUTING, SessionStatus.ESCALATED))
    sm.add_transition(Transition(
        SessionStatus.PROCESSING, SessionStatus.DRAFTING,
        guard=guard_classification_complete))
    sm.add_transition(Transition(
        SessionStatus.PROCESSING, SessionStatus.ESCALATED))
    sm.add_transition(Transition(
        SessionStatus.DRAFTING, SessionStatus.RESOLVED,
        guard=guard_draft_complete))
    for s in SessionStatus:
        sm.add_transition(Transition(s, SessionStatus.FAILED))
    return sm


## Step 2: The Memory Manager

### `memory/manager.py`

With the controller defining valid movement, the memory manager stores the information the system needs between steps. It separates short-term working context from longer-term records.

In [ ]:
# memory/manager.py
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional
from .base import MemoryInterface

@dataclass
class MemoryEntry:
    key: str
    value: Any
    memory_type: str       # 'working' | 'episodic' | 'semantic'
    created_at: datetime = field(default_factory=datetime.utcnow)
    valid_until: Optional[datetime] = None
    superseded_by: Optional[str] = None   # key of newer entry
    confidence: str = "high"              # high | medium | low

class MemoryManager:
    def __init__(
        self,
        working_store:  MemoryInterface,
        episodic_store: MemoryInterface,
        semantic_store: MemoryInterface
    ):
        self.working  = working_store
        self.episodic = episodic_store
        self.semantic = semantic_store

    # ── Working memory (session-scoped) ───────────────────────────────
    def store_working(self, key: str, value: Any) -> None:
        self.working.store(f"working:{key}", value)

    def get_working(self, key: str) -> Optional[Any]:
        return self.working.retrieve(f"working:{key}")

    # ── Episodic memory (customer history) ────────────────────────────
    def store_episode(self, customer_id: str, event: dict) -> None:
        key = f"episode:{customer_id}:{datetime.utcnow().isoformat()}"
        entry = MemoryEntry(
            key=key, value=event, memory_type="episodic"
        )
        self.episodic.store(key, entry)

    def get_customer_history(self, customer_id: str,
                              max_episodes: int = 5) -> list[MemoryEntry]:
        prefix = f"episode:{customer_id}:"
        keys = self.episodic.list_keys(prefix=prefix)
        entries = []
        for k in sorted(keys, reverse=True)[:max_episodes]:
            e = self.episodic.retrieve(k)
            if e and e.superseded_by is None:
                entries.append(e)
        return entries

    # ── Semantic memory (knowledge base) ──────────────────────────────
    def store_document(self, doc_id: str, content: str,
                        metadata: dict = None) -> None:
        # Check for superseded entries first
        existing = self.semantic.retrieve(f"doc:{doc_id}")
        if existing:
            existing.superseded_by = f"doc:{doc_id}:v_new"
            self.semantic.store(f"doc:{doc_id}:superseded", existing)
        entry = MemoryEntry(
            key=f"doc:{doc_id}",
            value={"content": content, "metadata": metadata or {}},
            memory_type="semantic"
        )
        self.semantic.store(f"doc:{doc_id}", entry)

    def search_knowledge(self, query: str,
                          top_k: int = 3) -> list[MemoryEntry]:
        raw = self.semantic.search(query, top_k=top_k)
        # Filter out superseded entries
        return [r for r in raw
                if isinstance(r, MemoryEntry) and r.superseded_by is None]


## Step 3: The Time Budget Enforcer

### `reasoning/budget.py`

Next, the time budget adds a hard operational boundary around the reasoning process. The system can now stop safely when it reaches its allowed time or step limits.

In [ ]:
# reasoning/budget.py
import time
from dataclasses import dataclass
from ..session import Session, SessionStatus

@dataclass
class BudgetStatus:
    elapsed_seconds: float
    step_count: int
    time_exceeded: bool
    step_exceeded: bool

    @property
    def exceeded(self) -> bool:
        return self.time_exceeded or self.step_exceeded

class BudgetEnforcer:
    def __init__(self, max_seconds: float, max_steps: int):
        self.max_seconds = max_seconds
        self.max_steps   = max_steps
        self._start      = time.time()
        self._steps      = 0

    def tick(self) -> BudgetStatus:
        """Call at the start of each step. Returns current budget status."""
        self._steps += 1
        elapsed = time.time() - self._start
        return BudgetStatus(
            elapsed_seconds=elapsed,
            step_count=self._steps,
            time_exceeded=elapsed > self.max_seconds,
            step_exceeded=self._steps > self.max_steps
        )

    def remaining_seconds(self) -> float:
        return max(0.0, self.max_seconds - (time.time() - self._start))


## Step 4: The Step Logger

### `reasoning/logger.py`

Once state, memory, and budgets are in place, the logger records what happened at each step. These records make the reasoning path easier to review, debug, and explain.

In [ ]:
# reasoning/logger.py
import json
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from typing import Any, Optional
from ..session import Session

@dataclass
class StepLog:
    session_id:    str
    step_number:   int
    step_name:     str
    inputs:        dict
    outputs:       dict
    status:        str         # success | failed | skipped
    confidence:    str         # high | medium | low
    cost_usd:      float
    latency_ms:    float
    timestamp:     str
    notes:         str = ""

class StepLogger:
    def __init__(self, session: Session,
                 persist_fn=None):
        """
        persist_fn: optional callable(StepLog) for external storage.
        If None, logs are stored in-memory on the session only.
        """
        self.session    = session
        self.persist_fn = persist_fn
        self._logs: list[StepLog] = []

    def log(self, step_name: str, inputs: dict, outputs: dict,
            status: str, confidence: str = "high",
            cost_usd: float = 0.0, latency_ms: float = 0.0,
            notes: str = "") -> StepLog:
        log = StepLog(
            session_id  = self.session.session_id,
            step_number = len(self._logs) + 1,
            step_name   = step_name,
            inputs      = inputs,
            outputs     = outputs,
            status      = status,
            confidence  = confidence,
            cost_usd    = cost_usd,
            latency_ms  = latency_ms,
            timestamp   = datetime.utcnow().isoformat()
        )
        self._logs.append(log)
        self.session.record_action(
            step_name,
            str(inputs)[:120],
            str(outputs)[:120],
            success=(status == "success"),
            cost=cost_usd,
            latency=latency_ms
        )
        if self.persist_fn:
            self.persist_fn(log)
        return log

    def trace(self) -> str:
        """Human-readable reasoning trace for this session."""
        lines = [f"Session {self.session.session_id} - ",
                 f"Ticket {self.session.ticket_id}", ""]
        for log in self._logs:
            status_icon = "ok" if log.status == "success" else "FAIL"
            lines.append(
                f"  Step {log.step_number}: {log.step_name} "
                f"[{status_icon}] conf={log.confidence} "
                f"${log.cost_usd:.4f} {log.latency_ms:.0f}ms"
            )
            if log.notes:
                lines.append(f"    Note: {log.notes}")
        lines.append("")
        lines.append(
            f"  Total: ${self.session.total_cost_usd:.4f} / "
            f"{self.session.total_latency_ms:.0f}ms"
        )
        return "\n".join(lines)


## Step 5: Wiring It All Together

### `main_ch3.py`

The final cell brings the controller, memory manager, budget, and logger into one flow. The result is a traceable reasoning loop that makes every major decision visible.

In [ ]:
# main_ch3.py
import os, time
import anthropic
from supportops_ai.config import DEFAULT_CONFIG
from supportops_ai.session import Session, SessionStatus
from supportops_ai.memory.in_memory import InMemoryStore
from supportops_ai.memory.manager import MemoryManager
from supportops_ai.reasoning.state_machine import (
    build_support_state_machine
)
from supportops_ai.reasoning.budget import BudgetEnforcer
from supportops_ai.reasoning.logger import StepLogger
from supportops_ai.router import route_request
from supportops_ai.services.classifier import classify_ticket
from supportops_ai.services.escalation import apply_escalation_rules
from supportops_ai.services.drafter import draft_response
from supportops_ai.tracker import SystemTrace

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
config = DEFAULT_CONFIG

# Shared stores (in production, these would be persistent)
memory = MemoryManager(
    working_store  = InMemoryStore(),
    episodic_store = InMemoryStore(),
    semantic_store = InMemoryStore()
)

def process_ticket_v3(ticket_id: str, raw_input: str,
                       customer_id: str = None) -> tuple[Session, str]:
    trace  = SystemTrace(ticket_id=ticket_id)
    session = Session(ticket_id=ticket_id, raw_input=raw_input)
    sm      = build_support_state_machine()
    budget  = BudgetEnforcer(max_seconds=60.0, max_steps=8)
    logger  = StepLogger(session)

    # ── Step 1: Route ────────────────────────────────────────────────
    status = budget.tick()
    sm.transition(session, SessionStatus.ROUTING)
    t0 = time.time()
    routing = route_request(client, raw_input, trace, config)
    session.routing_category = routing.category.value
    logger.log(
        "route",
        inputs  = {"input_len": len(raw_input)},
        outputs = {"category": routing.category.value,
                   "confidence": routing.confidence,
                   "immediate_escalation":
                       routing.requires_immediate_escalation},
        status  = "success",
        confidence = routing.confidence,
        latency_ms = (time.time()-t0)*1000
    )

    if routing.requires_immediate_escalation:
        sm.transition(session, SessionStatus.ESCALATED)
        session.escalated = True
        session.escalation_reason = "immediate_escalation_by_router"
        logger.log("escalate", {}, {"reason": session.escalation_reason},
                   "success")
        return session, logger.trace()

    # ── Step 2: Load customer history ────────────────────────────────
    if customer_id:
        history = memory.get_customer_history(customer_id)
        session.retrieved_context.extend([e.value for e in history])
        logger.log(
            "load_history",
            inputs  = {"customer_id": customer_id},
            outputs = {"episodes_loaded": len(history)},
            status  = "success"
        )

    # ── Step 3: Classify ─────────────────────────────────────────────
    status = budget.tick()
    if status.exceeded:
        session.status = SessionStatus.FAILED
        return session, logger.trace()

    ok, reason = sm.transition(session, SessionStatus.PROCESSING)
    if not ok:
        session.status = SessionStatus.FAILED
        logger.log("classify", {}, {"error": reason}, "failed")
        return session, logger.trace()

    t0 = time.time()
    clf = classify_ticket(client, raw_input, trace, config)
    session.classification = clf
    logger.log(
        "classify",
        inputs  = {"routing": session.routing_category},
        outputs = {"category": clf.category.value,
                   "urgency": clf.urgency.value,
                   "sentiment": clf.sentiment,
                   "review_required": clf.requires_human_review},
        status  = "success",
        latency_ms = (time.time()-t0)*1000
    )

    # ── Step 4: Escalation rules ─────────────────────────────────────
    session = apply_escalation_rules(session)
    logger.log(
        "escalation_check",
        inputs  = {"urgency": clf.urgency.value},
        outputs = {"escalated": session.escalated,
                   "reason": session.escalation_reason},
        status  = "success",
        confidence = "high",
        notes   = "Deterministic rules - no model call"
    )

    if session.escalated:
        sm.transition(session, SessionStatus.ESCALATED)
        # Store episode for future memory
        if customer_id:
            memory.store_episode(customer_id, {
                "ticket_id": ticket_id,
                "outcome": "escalated",
                "reason": session.escalation_reason
            })
        return session, logger.trace()

    # ── Step 5: Draft response ───────────────────────────────────────
    status = budget.tick()
    if status.exceeded:
        session.status = SessionStatus.FAILED
        return session, logger.trace()

    sm.transition(session, SessionStatus.DRAFTING)
    t0 = time.time()
    draft = draft_response(client, session, trace, config)
    session.draft_response = draft
    logger.log(
        "draft_response",
        inputs  = {"classification": clf.category.value},
        outputs = {"draft_len": len(draft), "preview": draft[:80]},
        status  = "success",
        latency_ms = (time.time()-t0)*1000
    )

    # ── Step 6: Transition to resolved ──────────────────────────────
    ok, reason = sm.transition(session, SessionStatus.RESOLVED)
    if not ok:
        session.status = SessionStatus.FAILED
        logger.log("resolve", {}, {"error": reason}, "failed")
        return session, logger.trace()

    # Store episode on success
    if customer_id:
        memory.store_episode(customer_id, {
            "ticket_id": ticket_id,
            "outcome": "resolved",
            "category": clf.category.value
        })

    logger.log("resolve", {}, {"status": "resolved"}, "success")
    return session, logger.trace()

# ── Run it ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    ticket = (
        "I upgraded my plan last week but my dashboard still shows "
        "the old limits. I've tried logging out and back in. "
        "This is blocking my team."
    )
    session, reasoning_trace = process_ticket_v3(
        "TKT-003", ticket, customer_id="C-8821"
    )
    print(reasoning_trace)
    print(f"\nFinal status: {session.status.value}")
    if session.draft_response:
        print(f"\nDraft:\n{session.draft_response}")
